In [1]:
#import packages
from pprint import pprint
import tensorflow as tf
import keras
import numpy as np
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.utility as utl
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.windowing as window
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.train_test_split as tts
import aneurysm_3Dsegmentation_in_CTA.model_utility.configuration as conf
import aneurysm_3Dsegmentation_in_CTA.model_utility.metrics as metrics
import segmentation_models_3D as sm3

I0000 00:00:1781012458.964836   47742 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Segmentation Models: using `tf.keras` framework.


In [3]:
# loading data tensors
dataSource = "data/CTA nii" 

smallAneurysm , mediumAneurysm , largeAneurysm = tts.dataSplitPerSize(dataSource)

trainSet , testSet = tts.dataSplitPerSample(
    [
        smallAneurysm ,
        mediumAneurysm ,
        largeAneurysm
    ] ,
    testRatio=0.2 ,
    seed=42
)

imgTrainSet , labelTrainSet = tts.dataTensorLoading(trainSet)
imgTestSet  , labelTestSet = tts.dataTensorLoading(testSet)


In [ ]:
# check for image pairs
for name in zip(imgTrainSet , labelTrainSet) :
    print(name[0])
    print(name[1])
    print("============")
print("//////////////////////////////////////////////// test ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgTestSet , labelTestSet) :
    print(name[0])
    print(name[1])
    print("============")

In [21]:
# pipeline configuring

geo      = utl.randomGeo(p=1)
crop     = utl.volume_crop((128 , 128 , 128))
tile     = utl.tile(
    tile_dim=[1 , 1 , 1 , 1 , 3]
)

setShape = utl.setShape([None , 128 , 128 , 128 , 3])

windower = window.randomMultiWindowStackig(
    default = (200 , 620) ,
    wlRange=(170 , 225) ,
    wwRange=(600 , 650) ,
    p_wl=1 ,
    p_ww=1
)

# wrapping
@tf.py_function(Tout=[tf.float64 , tf.float64])
def rimg(imgPath , labelPath) :
    return utl.read_img(imgPath , labelPath)
def read_img(img , label) :
    imglbl = rimg(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label

@tf.py_function(Tout=[tf.float64 , tf.float64])
def rotate(img , label) :
    img , label = geo.rot(
        img , 
        label ,
        imgOrder=1 ,
        lblOrder=0 ,
        imgCval=-1024 ,
        lblCval=0
    )
    img , label = geo.flip(
        img , 
        label
    )
    return img , label
def rot(img , label) :
    imglbl = rotate(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label



In [22]:
# dataloaders
dataloaderTrain = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet))
    .map(
        read_img , 
        num_parallel_calls=4
    )
    .map(
        crop.cropping ,
        num_parallel_calls=4
    )

    #.cache("myCacheTrain")
    #.shuffle(buffer_size=100)
    
    .map(
        rot ,
        num_parallel_calls=4
    )
    .map(
        windower.WindowStacking ,
        num_parallel_calls=4
    )

    .batch(batch_size=2)
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

dataloaderValid = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet))

    .map(
        read_img , 
        num_parallel_calls=4
    )
    .map(
        crop.cropping ,
        num_parallel_calls=4
    )

    #.cache("myCacheValid")
    .batch(batch_size=2)
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

In [ ]:
cnt=0
# vectorize data model testing

for data in dataloaderTrain.take(10) :
    print(data[0].shape)
    print(data[0].dtype)
    print(data[1].shape)
    print(data[1].dtype)

In [23]:
#loss

binaryFocalLoss = sm3.losses.binary_focal_loss
diceLoss        = sm3.losses.dice_loss 
weightedBinaryFocalDiceLoss = conf.WeightedSumOfLosses(binaryFocalLoss , diceLoss , alpha=0.8)

model = sm3.Unet(
    backbone_name="seresnet18" , 
    input_shape=(128 , 128 , 128 , 3) , 
    classes=1 , 
    activation="sigmoid" ,
    encoder_weights="imagenet" ,
    encoder_freeze=True ,
    decoder_block_type="transpose" ,
    encoder_features=conf.encoderF_d4
)

# model unfreezing
model = conf.unfreeze_model(
    model , 
    conf.unfreeze34_border , 
    keras.src.layers.normalization.batch_normalization.BatchNormalization
)

In [ ]:
print(model.summary())

In [24]:
# compilation
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3) ,
    loss = weightedBinaryFocalDiceLoss ,
    metrics=[
        metrics.V_Recall ,
        metrics.dice
    ]
)

In [25]:
sim = np.random.random(
    size=(2 , 128 , 128 , 128 , 3) ,
)
lab = np.random.random(
    size=(2 , 128 , 128 , 128 , 1) ,
)
lab = np.where(lab>=0.5 , 1.0 , 0.0)

# model training
history = model.fit(
    x=dataloaderTrain ,
    epochs=5 ,
    #validation_data=dataloaderValid
)

Epoch 1/5
33/95 ━━━━━━━━━━━━━━━━━━━━ 8:32 8s/step - dice: 3.6932e-04 - loss: 0.2654 - v__recall: 15.9528

W0000 00:00:1781014643.376262   56214 cpu_allocator_impl.cc:82] Allocation of 1514143744 exceeds 10% of free system memory.
W0000 00:00:1781014645.584263   56214 cpu_allocator_impl.cc:82] Allocation of 1514143744 exceeds 10% of free system memory.


34/95 ━━━━━━━━━━━━━━━━━━━━ 8:24 8s/step - dice: 3.6365e-04 - loss: 0.2641 - v__recall: 15.6214

W0000 00:00:1781014652.081596   56214 cpu_allocator_impl.cc:82] Allocation of 1514143744 exceeds 10% of free system memory.
W0000 00:00:1781014655.464878   56214 cpu_allocator_impl.cc:82] Allocation of 1514143744 exceeds 10% of free system memory.


37/95 ━━━━━━━━━━━━━━━━━━━━ 8:10 8s/step - dice: 3.4768e-04 - loss: 0.2606 - v__recall: 14.7138

W0000 00:00:1781014685.326253   56216 cpu_allocator_impl.cc:82] Allocation of 1512046592 exceeds 10% of free system memory.


94/95 ━━━━━━━━━━━━━━━━━━━━ 9s 10s/step - dice: 0.0013 - loss: 0.2313 - v__recall: 7.4409     

E0000 00:00:1781015290.393450   47864 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1781015298.008810   47864 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1781015303.281830   47864 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1781015303.875756   47864 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


95/95 ━━━━━━━━━━━━━━━━━━━━ 1064s 10s/step - dice: 0.0019 - loss: 0.2308 - v__recall: 7.3737
Epoch 2/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 990s 10s/step - dice: 0.1558 - loss: 0.1977 - v__recall: 19.7810
Epoch 3/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 978s 10s/step - dice: 0.6051 - loss: 0.1346 - v__recall: 51.7343
Epoch 4/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 1005s 10s/step - dice: 0.8106 - loss: 0.0985 - v__recall: 65.7530
Epoch 5/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 1030s 10s/step - dice: 0.8228 - loss: 0.0943 - v__recall: 65.2386
